# Denoise: find the noisy and music chunks, compare DeepFilterNet3 and MossFormerGAN
Works on the dataset from `07_chunk_transcribe_dataset` (`dataset/`) and never changes it. All folders are in the run folder that `RUN` in `pipeline_paths.py` sets (the project root, or `runs/<RUN>/`).

**Kernel: `.venv-denoise` (Python 3.11)**, not the project's `.venv`: DeepFilterNet and ClearerVoice need numpy < 2 and an older torch. It was made with
```
uv venv .venv-denoise --python 3.11
uv pip install --python .venv-denoise/bin/python "torch==2.5.1" "torchaudio==2.5.1" "torchvision==0.20.1" "numpy<2" deepfilternet clearvoice "transformers<5" ipykernel matplotlib tqdm
```
On the DGX Spark (ARM, CUDA 13) the env is the same but with its torch:
```
uv venv .venv-denoise --python 3.11
uv pip install --python .venv-denoise/bin/python --extra-index-url https://download.pytorch.org/whl/cu130 --index-strategy unsafe-best-match "torch==2.14.1+cu130" "torchaudio==2.11.0+cu130" "numpy<2" deepfilternet clearvoice "transformers<5" ipykernel matplotlib tqdm
```
and it runs without Jupyter, so it keeps going after the terminal closes: `python scripts/run_notebook.py src/data_preprocessing/08_denoise.ipynb` inside `tmux`.

Models, downloaded on first use: DeepFilterNet3 (`~/Library/Caches/DeepFilterNet`), MossFormerGAN_SE_16K (`~/.cache/clearvoice`), the AudioSet classifier AST (Hugging Face cache).

Run the cells from top to bottom. Steps 1 and 4 skip what they have already done, so after a stop you can run them again.

**1. Scan** (`outputs/denoise/scores.json`). Every chunk gets two numbers:
- **music**: the AudioSet "Music" probability from AST, for the chunk's most musical 10 s.
- **background**: how loud the steady background is in the pauses, in dB below the speech.
  - DeepFilterNet3 finds the pauses: frames where its output is 30 dB below the speech level.
  - The background is the quiet end (20th percentile) of the original audio in those pauses. Hiss, hum, room noise or a music bed count; a laugh or a "hm" in a pause doesn't.
  - Clean studio podcasts sit at −50 dB and below; noisy ones at around −30 dB.
- Takes about 45 min for the whole dataset: DeepFilterNet3 runs on the CPU while AST runs on the GPU.
- The chunks are scanned in random order. If you stop early, the scanned part already covers every podcast, and steps 2–3 work on it.

**2. Find** (`outputs/denoise/flagged.json`)
- A chunk with music ≥ `MUSIC_MIN` is **music**.
- Otherwise, a background ≥ `NOISY_DB` makes it **noisy**.
- Everything else is **clean**.
- A chunk with less than `MIN_PAUSE_SEC` of pause gets its speaker's median background (same mic, same room).

**3. Before/after samples** (`outputs/denoise/samples/`; open `index.html` in a browser to listen). `N_SAMPLES` music chunks and `N_SAMPLES` noisy chunks, spread from mild to strong, each as:
- `before.wav`: the original;
- `dfn3.wav`: DeepFilterNet3, which works at 48 kHz itself;
- `mossformergan.wav`: MossFormerGAN below 7 kHz and DeepFilterNet3 above it, at 48 kHz;
- `mossformergan_plain.wav`: MossFormerGAN alone, resampled to 48 kHz. It works at 16 kHz only, so this file has nothing above 8 kHz.

All files are 48 kHz 16-bit, like the dataset.

**4. Denoise the dataset** (`dataset_clean/`)
- `USE` picks a model for each category. It is set to the choice made after listening on 2026-11-25 (MossFormerGAN for music, DeepFilterNet3 for noisy), so step 4 runs without stopping; set `USE = None` to stop after the samples. Every music or noisy chunk is denoised with its model.
- Clean chunks are hard links to the originals, so they take no extra disk space.
- `metadata.json` is the same as the dataset's, plus `"denoised": <model>` on the chunks that were denoised.
- Speed:
  - DeepFilterNet3 takes about 1 s per minute of audio.
  - MossFormerGAN takes about 0.5 s per second of audio on the GPU, so 5 h of flagged audio is about 2.5 h of running. Step 2 prints the hours per category.
- DeepFilterNet3 takes a lone hum ("Hm", "mm") for noise and removes it. Step 4 leaves out chunks where the model removed more than it kept (`MAX_REMOVED_DB`).

In [ ]:
import contextlib
import html
import json
import os
import random
import shutil
import time
import warnings
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torchaudio.functional as AF
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

import importlib
import pipeline_paths as P
importlib.reload(P)   # a change of RUN in pipeline_paths.py counts without restarting the kernel

root = P.ROOT
dataset_dir = P.DATASET                     # input, from 07: metadata.json + chunks/; not changed here
work_dir = P.OUTPUTS / "denoise"            # scores.json, flagged.json, samples/
out_dir = P.DATASET_CLEAN                   # step 4
print(P.describe())
meta = json.load(open(dataset_dir / "metadata.json", encoding="utf-8"))
chunks = meta["chunks"]
SR = meta["sample_rate"]
assert SR == 48000, "DeepFilterNet3 works at 48 kHz"
print(f"{len(chunks)} chunks, {sum(c['duration'] for c in chunks) / 3600:.1f} h; torch {torch.__version__}")

In [ ]:
ONLY = None             # e.g. ["019", "023"]: only these podcasts (numbers as in metadata.json); None = all

# steps 1-2: what counts as music / noisy
MUSIC_MIN = 0.2         # AudioSet "Music" probability at or above this = music
NOISY_DB = -35.0        # steady background in the pauses, dB relative to the speech; at or above this = noisy
MIN_PAUSE_SEC = 0.3     # less pause than this in a chunk: it gets its speaker's median background
SAVE_EVERY = 200

# step 3: before/after samples
N_SAMPLES = 6           # per category, spread from mild to strong
SAMPLE_SEC = (4, 20)    # sample length range, seconds
SEED = 0

# MossFormerGAN works at 16 kHz only. "mossformergan" takes MossFormerGAN below CROSSOVER_HZ and DeepFilterNet3
# above it; "mossformergan_plain" is MossFormerGAN alone (nothing above 8 kHz).
CROSSOVER_HZ = (6500, 7500)

# step 4: a model per category: "dfn3", "mossformergan", "mossformergan_plain", or None to keep the original.
# This is the choice made after listening to the samples on 2026-11-25. USE = None stops after step 3.
USE = {"music": "mossformergan", "noisy": "dfn3"}
MAX_REMOVED_DB = 0.0    # leave out a chunk when the model removed more energy than it kept (a lone "Hm", or junk)

RESAMPLE = dict(lowpass_filter_width=64, rolloff=0.99, resampling_method="sinc_interp_kaiser", beta=14.769656459379492)
MODELS = ("dfn3", "mossformergan", "mossformergan_plain")
assert USE is None or (set(USE) <= {"music", "noisy"} and all(m is None or m in MODELS for m in USE.values())), USE

## Models and measurements

In [ ]:
dev = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
selected = [c for c in chunks if ONLY is None or c["speaker"][:3] in ONLY]

# DeepFilterNet3: 48 kHz, on the CPU (on the GPU when there is CUDA)
try:  # DeepFilterNet imports torchaudio.backend.common, which torchaudio >= 2.9 no longer has; it is not used here
    import torchaudio.backend.common  # noqa: F401
except ImportError:
    import sys
    import types

    for _name in ("torchaudio.backend", "torchaudio.backend.common"):
        sys.modules[_name] = types.ModuleType(_name)
    sys.modules["torchaudio.backend.common"].AudioMetaData = object
from df.enhance import enhance, init_df

dfn, df_state, _ = init_df(log_level="ERROR")
assert df_state.sr() == SR


def dfn3(x):
    with torch.no_grad():
        return enhance(dfn, df_state, torch.from_numpy(x)[None], pad=True)[0].numpy()


# MossFormerGAN_SE_16K from ClearerVoice. ClearVoice downloads it into ./checkpoints, so it is loaded from the cache dir.
cv_cache = Path.home() / ".cache" / "clearvoice"
cv_cache.mkdir(parents=True, exist_ok=True)
with contextlib.chdir(cv_cache):
    from clearvoice import ClearVoice
    mf_net = ClearVoice(task="speech_enhancement", model_names=["MossFormerGAN_SE_16K"]).models[0].model
mf_net = mf_net.eval().to(dev)
MF_WIN = torch.hamming_window(400, periodic=True)


def mf_forward(spec):
    """MossFormerGAN's forward pass (SyncANet.forward) without complex tensors: torch.angle does not run on MPS."""
    mag = torch.sqrt(spec[:, :1] ** 2 + spec[:, 1:] ** 2)
    safe = mag.clamp_min(1e-12)
    cos = torch.where(mag > 0, spec[:, :1] / safe, torch.ones_like(mag))
    sin = torch.where(mag > 0, spec[:, 1:] / safe, torch.zeros_like(mag))
    h = mf_net.dense_encoder(torch.cat([mag, spec], 1))
    for block in mf_net.blocks:
        h = block(h)
    out_mag = mf_net.mask_decoder(h) * mag
    c = mf_net.complex_decoder(h)
    return out_mag * cos + c[:, :1], out_mag * sin + c[:, 1:]


@torch.no_grad()
def mf_once(x, norm):
    """One window (16 kHz) through MossFormerGAN, as ClearVoice does it: STFT 400/100 with |S| ** 0.3 on the CPU,
    the network on the GPU."""
    n = len(x)
    x = torch.cat([x, x[:(-n) % 100]]) * norm
    s = torch.stft(x[None], 400, 100, 400, window=MF_WIN, center=True, onesided=True, return_complex=True)
    s = s * s.abs().clamp_min(1e-12) ** -0.7
    re, im = mf_forward(torch.stack([s.real, s.imag], 1).permute(0, 1, 3, 2).to(dev))
    re, im = (t.permute(0, 1, 3, 2).squeeze(1).float().cpu() for t in (re, im))
    mag = torch.sqrt(re ** 2 + im ** 2).clamp_min(1e-12)
    y = torch.istft(torch.complex(re, im) * mag ** (1 / 0.3 - 1), 400, 100, 400, window=MF_WIN,
                    center=True, onesided=True)
    return (y[0] / norm)[:n]


@torch.no_grad()
def mfgan16(x48, window=160000, stride=120000):
    """MossFormerGAN on a 48 kHz chunk, returns 16 kHz. Long chunks: 10 s windows 7.5 s apart, like ClearVoice."""
    x = AF.resample(torch.from_numpy(x48), SR, 16000, **RESAMPLE)
    norm = torch.sqrt(len(x) / (x ** 2).sum().clamp_min(1e-12))
    if len(x) <= window:
        out = mf_once(x, norm)
    else:
        out, give, i = torch.zeros(len(x)), (window - stride) // 2, 0
        while i + window <= len(x):
            y = mf_once(x[i:i + window], norm)
            if i == 0:
                out[:window - give] = y[:window - give]
            else:
                out[i + give:i + window - give] = y[give:window - give]
            i += stride
        if i < len(x):
            out[i:] = mf_once(x[i - give:], norm)[give:]
    if dev.type == "mps":
        torch.mps.empty_cache()   # otherwise the GPU cache grows with every new chunk length, into swap, and slows down
    return out.numpy()


def up48(y16, n):
    y = AF.resample(torch.from_numpy(y16), 16000, SR, **RESAMPLE).numpy()
    return np.pad(y, (0, max(0, n - len(y))))[:n]


def crossover(low, high):
    """`low` below CROSSOVER_HZ, `high` above it, with a raised-cosine crossover (zero phase, one FFT)."""
    f = np.fft.rfftfreq(len(high), 1 / SR)
    a, b = CROSSOVER_HZ
    w = 0.5 - 0.5 * np.cos(np.pi * np.clip((b - f) / (b - a), 0, 1))
    H, L = np.fft.rfft(high.astype(np.float64)), np.fft.rfft(low.astype(np.float64))
    return np.fft.irfft(H + w * (L - H), len(high)).astype(np.float32)


def run_model(name, x, d=None):
    """48 kHz chunk -> denoised 48 kHz chunk. d: DeepFilterNet3's output of x, if it is already there."""
    d = dfn3(x) if d is None and name != "mossformergan_plain" else d
    if name == "dfn3":
        return d
    m = up48(mfgan16(x), len(x))
    return m if name == "mossformergan_plain" else crossover(m, d)


# AST, AudioSet classifier (527 sound classes), on the GPU
from transformers import ASTFeatureExtractor, ASTForAudioClassification

AST_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
ast_fe = ASTFeatureExtractor.from_pretrained(AST_NAME)
ast = ASTForAudioClassification.from_pretrained(AST_NAME).eval().to(dev)
MUSIC = ast.config.label2id["Music"]


def music_prob(x48, n=160000):
    """AudioSet "Music" probability of the chunk's most musical 10 s window."""
    x = AF.resample(torch.from_numpy(x48), SR, 16000).numpy()
    starts = [0] if len(x) <= n else [*range(0, len(x) - n, n), len(x) - n]
    feats = ast_fe([x[s:s + n] for s in starts], sampling_rate=16000, return_tensors="pt")["input_values"]
    with torch.inference_mode():
        return float(torch.sigmoid(ast(feats.to(dev)).logits[:, MUSIC]).max())


FRAME = SR // 50        # 20 ms
PAUSE_GAP_DB = 30.0     # changing it needs a new scan (delete scores.json)


def frame_db(x):
    m = len(x) // FRAME
    p = (x[:m * FRAME].reshape(m, FRAME).astype(np.float64) ** 2).mean(1)
    return 10 * np.log10(np.maximum(p, 1e-12))


def pause_frames(d):
    """Frames where DeepFilterNet3's output d is PAUSE_GAP_DB below its speech level (its loud frames)."""
    dd = frame_db(d)
    return dd < np.percentile(dd, 90) - PAUSE_GAP_DB


def background(v, pause):
    """Steady background of v in the pause frames (their quiet end), dB relative to v's speech level."""
    vd = frame_db(v)
    return round(float(np.percentile(vd[pause], 20) - np.percentile(vd, 90)), 1) if pause.any() else None


def removed_db(x, v):
    """Energy the model took out of x, relative to what it kept: above 0 = it removed more than it kept."""
    return round(float(10 * np.log10(max(((x - v) ** 2).sum(), 1e-12) / max((v ** 2).sum(), 1e-12))), 1)


def fmt(v, spec="6.1f"):
    return format(v, spec) if v is not None else "   n/a"


def save_json(path, obj, indent=1):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=indent), encoding="utf-8")
    os.replace(tmp, path)


def write_wav(path, v):
    path.parent.mkdir(parents=True, exist_ok=True)
    peak = float(np.abs(v).max())
    sf.write(path, v * (0.999 / peak) if peak > 0.999 else v, SR, subtype="PCM_16")


print(f"models loaded; GPU: {dev}; {len(selected)} chunks selected")

## 1. Scan every chunk

In [ ]:
scores_path = work_dir / "scores.json"
scores = json.load(open(scores_path, encoding="utf-8")) if scores_path.exists() else {}
order = selected[:]
random.Random(SEED).shuffle(order)   # random order: a partial scan already covers every podcast
todo = [c for c in order if scores.get(c["file"], {}).get("duration") != c["duration"]]
print(f"{len(selected) - len(todo)} of {len(selected)} chunks already scanned; "
      f"{len(todo)} to go ({sum(c['duration'] for c in todo) / 3600:.1f} h of audio)")


def scan(c, pool):
    x, sr = sf.read(dataset_dir / c["file"], dtype="float32")
    assert sr == SR and x.ndim == 1, (c["file"], sr, x.shape)
    music = pool.submit(music_prob, x)       # AST on the GPU while DeepFilterNet3 runs on the CPU
    d = dfn3(x)
    pause = pause_frames(d)
    return dict(duration=c["duration"], music=round(music.result(), 3), bg=background(x, pause),
                pause=round(pause.sum() / 50, 2), removed=removed_db(x, d))


failed = {}
t0 = time.time()
try:
    with ThreadPoolExecutor(1) as pool:
        for k, c in enumerate(tqdm(todo, desc="scan"), 1):
            try:
                scores[c["file"]] = scan(c, pool)
            except Exception as e:
                failed[c["file"]] = repr(e)
            if k % SAVE_EVERY == 0:
                save_json(scores_path, scores)
finally:
    save_json(scores_path, scores)
print(f"done in {(time.time() - t0) / 60:.1f} min; {len(failed)} failed")
for f, e in list(failed.items())[:10]:
    print("  ", f, e)

## 2. Find the music and noisy chunks

In [ ]:
def measured(s):
    return s["bg"] is not None and s["pause"] >= MIN_PAUSE_SEC


have = [c for c in selected if scores.get(c["file"], {}).get("duration") == c["duration"]]
spk_bg = defaultdict(list)
for c in have:
    if measured(scores[c["file"]]):
        spk_bg[c["speaker"]].append(scores[c["file"]]["bg"])
spk_bg = {k: float(np.median(v)) for k, v in spk_bg.items()}

cat, bg = {}, {}
for c in have:
    s = scores[c["file"]]
    bg[c["file"]] = s["bg"] if measured(s) else spk_bg.get(c["speaker"])
    if s["music"] >= MUSIC_MIN:
        cat[c["file"]] = "music"
    elif bg[c["file"]] is not None and bg[c["file"]] >= NOISY_DB:
        cat[c["file"]] = "noisy"
    else:
        cat[c["file"]] = "clean"

if len(have) < len(selected):
    print(f"only {len(have)} of {len(selected)} chunks are scanned: run step 1 again to finish. The numbers are for those.\n")
print(f"{'':6} {'chunks':>7} {'hours':>6}")
for k in ["music", "noisy", "clean"]:
    part = [c for c in have if cat[c["file"]] == k]
    print(f"{k:6} {len(part):7d} {sum(c['duration'] for c in part) / 3600:6.2f}")
n_est = sum(1 for c in have if not measured(scores[c["file"]]))
print(f"({n_est} chunks have less than {MIN_PAUSE_SEC} s of pause: their background is their speaker's median)")

print(f"\n{'podcast':52} {'chunks':>6} {'music':>6} {'noisy':>6} {'median bg':>10}")
by_pod = defaultdict(list)
for c in have:
    by_pod[c["speaker"][:3]].append(c)
for num, cs in sorted(by_pod.items()):
    cnt = Counter(cat[c["file"]] for c in cs)
    med = np.median([b for c in cs if (b := bg[c["file"]]) is not None] or [np.nan])
    print(f"{num} {meta['podcasts'][num][:48]:48} {len(cs):6d} {cnt['music']:6d} {cnt['noisy']:6d} {med:8.1f} dB")

by_file = {c["file"]: c for c in chunks}
for k, key in [("music", lambda f: -scores[f]["music"]), ("noisy", lambda f: -bg[f])]:
    print(f"\nmost {k}:")
    for f in sorted((f for f in cat if cat[f] == k), key=key)[:8]:
        c, s = by_file[f], scores[f]
        print(f"  {f:32} music {s['music']:.2f}  bg {fmt(bg[f])} dB  {c['duration']:5.1f} s  {c['text'][:60]}")

save_json(work_dir / "flagged.json", dict(
    settings=dict(MUSIC_MIN=MUSIC_MIN, NOISY_DB=NOISY_DB, MIN_PAUSE_SEC=MIN_PAUSE_SEC),
    music=sorted(f for f in cat if cat[f] == "music"), noisy=sorted(f for f in cat if cat[f] == "noisy")))

## 3. Before/after samples
Every number in the table is measured on that file:
- **bg**: the background in the pauses, in dB below the speech. Lower is cleaner.
- **music**: the AST music probability.
- **removed**: the energy the model took out, relative to what it kept.

To listen to all of them, open `outputs/denoise/samples/index.html` in a browser, or play one in the next cell.

In [ ]:
import matplotlib.pyplot as plt

NAMES = ["before", "dfn3", "mossformergan", "mossformergan_plain"]
samples_dir = work_dir / "samples"


def pick(k, key):
    """N_SAMPLES chunks of category k, evenly spread over their sorted scores: from mild to strong."""
    cs = sorted((c for c in have if cat[c["file"]] == k and SAMPLE_SEC[0] <= c["duration"] <= SAMPLE_SEC[1]
                 and scores[c["file"]]["removed"] < MAX_REMOVED_DB and (k == "music" or measured(scores[c["file"]]))),
                key=key)
    if len(cs) <= N_SAMPLES:
        return cs
    return [cs[i] for i in np.unique(np.linspace(0, len(cs) - 1, N_SAMPLES).round().astype(int))]


picks = {"music": pick("music", lambda c: scores[c["file"]]["music"]),
         "noisy": pick("noisy", lambda c: scores[c["file"]]["bg"])}
shutil.rmtree(samples_dir, ignore_errors=True)
rows = []
for k, cs in picks.items():
    for i, c in enumerate(tqdm(cs, desc=k), 1):
        x, _ = sf.read(dataset_dir / c["file"], dtype="float32")
        d = dfn3(x)
        m = up48(mfgan16(x), len(x))
        versions = dict(before=x, dfn3=d, mossformergan=crossover(m, d), mossformergan_plain=m)
        pause = pause_frames(d)
        folder = samples_dir / k / f"{i:02d}_{Path(c['file']).stem}"
        row = dict(category=k, file=c["file"], folder=folder.relative_to(samples_dir).as_posix(),
                   duration=c["duration"], text=c["text"])
        fig, axes = plt.subplots(1, 4, figsize=(18, 2.8), sharey=True)
        for ax, (name, v) in zip(axes, versions.items()):
            write_wav(folder / f"{name}.wav", v)
            row[name] = dict(bg=background(v, pause), music=round(music_prob(v), 3),
                             removed=None if name == "before" else removed_db(x, v))
            ax.specgram(v, NFFT=2048, Fs=SR, noverlap=1536, cmap="magma", vmin=-150)
            ax.set_ylim(0, 16000)
            ax.set_title(f"{name}: bg {fmt(row[name]['bg'], '.1f')} dB, music {row[name]['music']:.2f}", fontsize=9)
        axes[0].set_ylabel("Hz")
        fig.tight_layout()
        fig.savefig(folder / "spectrogram.png", dpi=70)
        plt.close(fig)
        rows.append(row)
save_json(samples_dir / "samples.json", rows)

# index.html: players and spectrograms of every sample
cells_html = []
for r in rows:
    players = "".join(
        f"<td><b>{n}</b><br><audio controls preload='none' src='{r['folder']}/{n}.wav'></audio><br>"
        f"bg {fmt(r[n]['bg'], '.1f')} dB · music {r[n]['music']:.2f}"
        + (f" · removed {r[n]['removed']:.1f} dB" if r[n]["removed"] is not None else "") + "</td>" for n in NAMES)
    cells_html.append(f"<tr><td colspan=4><h3>{r['category']}: {html.escape(r['file'])} · {r['duration']:.1f} s</h3>"
                      f"<p>{html.escape(r['text'])}</p><img src='{r['folder']}/spectrogram.png' width=100%></td></tr>"
                      f"<tr>{players}</tr>")
(samples_dir / "index.html").write_text(
    "<!DOCTYPE html><meta charset='utf-8'><title>Denoise samples</title>"
    "<style>body{font-family:system-ui;margin:24px}td{vertical-align:top;padding:4px 12px 16px 0}"
    "h3{margin:24px 0 4px}p{margin:0 0 8px;color:#444}</style>"
    f"<h1>Before / after</h1><p>bg = background in the pauses, dB below the speech (lower is cleaner). "
    f"mossformergan = MossFormerGAN below {CROSSOVER_HZ[0] / 1000:g}-{CROSSOVER_HZ[1] / 1000:g} kHz + DeepFilterNet3 above; "
    f"mossformergan_plain = MossFormerGAN alone (16 kHz model, nothing above 8 kHz).</p>"
    f"<table>{''.join(cells_html)}</table>", encoding="utf-8")

print(f"{'':6} {'file':30}" + "".join(f"{n:>22}" for n in NAMES))
print(f"{'':37}" + "".join(f"{'bg dB  music':>22}" for n in NAMES))
for r in rows:
    print(f"{r['category']:6} {Path(r['file']).stem:24} {r['duration']:4.1f}s"
          + "".join(f"{fmt(r[n]['bg']):>15} {r[n]['music']:5.2f}" for n in NAMES))
print(f"\nopen in a browser: {samples_dir / 'index.html'}")

### Play one sample here
`play("music", 1)` plays the first music sample. Each one adds about 4 MB to the notebook, so use the `index.html` for the rest.

In [ ]:
from IPython.display import Audio, Image, display


def play(k, i):
    r = [r for r in json.load(open(samples_dir / "samples.json", encoding="utf-8")) if r["category"] == k][i - 1]
    print(f"{r['file']}  {r['duration']:.1f} s  {r['text']}")
    display(Image(str(samples_dir / r["folder"] / "spectrogram.png")))
    for n in NAMES:
        print(f"{n}: bg {fmt(r[n]['bg'], '.1f')} dB, music {r[n]['music']:.2f}")
        display(Audio(str(samples_dir / r["folder"] / f"{n}.wav")))


play("noisy", 1)

## 4. Denoise the dataset
Set `USE` in the config cell first. It needs the whole dataset scanned (`ONLY = None`, step 1 finished).

In [ ]:
missing = [c for c in chunks if c["file"] not in cat]
if not USE:
    print("Listen to the samples first, then set USE in the config cell and run steps 2 and 4 again.")
elif ONLY is not None or missing:
    print(f"{len(missing)} chunks are not scanned: set ONLY = None, run step 1 and step 2, then this cell.")
else:
    log_path = out_dir / "denoise_log.json"   # file -> {"model": "" (= original kept) or a model, "dropped": bool}
    log = json.load(open(log_path, encoding="utf-8")) if log_path.exists() else {}
    want = {c["file"]: USE.get(cat[c["file"]]) or "" for c in chunks}
    done = lambda f: f in log and log[f]["model"] == want[f] and (log[f]["dropped"] or (out_dir / f).exists())
    todo = [c for c in chunks if not done(c["file"])]
    print(f"{len(chunks) - len(todo)} chunks already done; {len(todo)} to go, "
          f"{sum(c['duration'] for c in todo if want[c['file']]) / 3600:.2f} h of them to denoise")
    (out_dir / "chunks").mkdir(parents=True, exist_ok=True)
    (out_dir / "metadata.json").unlink(missing_ok=True)   # written last: no metadata.json = not finished
    t0 = time.time()
    try:
        for k, c in enumerate(tqdm(todo, desc="denoise"), 1):
            f, model = c["file"], want[c["file"]]
            src, dst = dataset_dir / f, out_dir / f
            dst.unlink(missing_ok=True)
            if not model:
                try:
                    os.link(src, dst)                 # unchanged chunk: a hard link, no extra disk space
                except OSError:
                    shutil.copy2(src, dst)
                log[f] = dict(model="", dropped=False)
            else:
                x, _ = sf.read(src, dtype="float32")
                v = run_model(model, x)
                r = removed_db(x, v)
                if r <= MAX_REMOVED_DB:
                    write_wav(dst, v)
                log[f] = dict(model=model, dropped=r > MAX_REMOVED_DB, removed=r)
            if k % SAVE_EVERY == 0:
                save_json(log_path, log)
    finally:
        save_json(log_path, log)
    kept = [dict(c, denoised=log[c["file"]]["model"]) if log[c["file"]]["model"] else c
            for c in chunks if not log[c["file"]]["dropped"]]
    names = {c["file"] for c in kept}
    for p in (out_dir / "chunks").glob("*.wav"):      # dropped now, or left from an older run
        if f"chunks/{p.name}" not in names:
            p.unlink()
    save_json(out_dir / "metadata.json", dict(meta, chunks=kept, denoise=dict(
        use=USE, MUSIC_MIN=MUSIC_MIN, NOISY_DB=NOISY_DB, CROSSOVER_HZ=CROSSOVER_HZ)), indent=2)
    print(f"done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# check and summary
if not (out_dir / "metadata.json").exists():
    print("no dataset_clean/metadata.json yet: step 4 has not run")
else:
    m2 = json.load(open(out_dir / "metadata.json", encoding="utf-8"))
    log = json.load(open(out_dir / "denoise_log.json", encoding="utf-8"))
    bad = [c["file"] for c in m2["chunks"] if sf.info(out_dir / c["file"]).frames != sf.info(dataset_dir / c["file"]).frames]
    print("wav lengths equal to the originals:", "all" if not bad else f"NO, {len(bad)} differ: {bad[:5]}")
    print(f"{len(m2['chunks'])} chunks, {sum(c['duration'] for c in m2['chunks']) / 3600:.2f} h")
    for model, n in Counter(c.get("denoised", "original") for c in m2["chunks"]).most_common():
        h = sum(c["duration"] for c in m2["chunks"] if c.get("denoised", "original") == model) / 3600
        print(f"  {model:20} {n:5d} chunks, {h:5.2f} h")
    dropped = sorted(f for f, v in log.items() if v["dropped"] and f in by_file)
    print(f"left out (model removed more than it kept): {len(dropped)}")
    for f in dropped[:20]:
        print(f"  {f}  {log[f]['removed']:+.1f} dB  {by_file[f]['text'][:60]}")